# 169 — IA neuro-simbólica

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — Taxonomía de Kautz

a) **Tipo 2, Symbolic[Neuro]**: el marco de control es simbólico (MCTS) y la red
   es una subrutina de evaluación.
b) **Tipo 6, Neuro[Symbolic]**: la red (LLM) decide invocar un motor exacto (el
   intérprete); el control lo tiene la parte neural.
c) **Tipo 4, Symbolic → Neuro**: el conocimiento simbólico se compila en el
   entrenamiento como restricción; en inferencia solo corre la red.
d) **Tipo 1, Symbolic Neuro symbolic**: entrada y salida simbólicas (texto), red
   en el medio, sin motor simbólico explícito.

La diferencia clave entre a) y b) es *quién controla el flujo*: en AlphaGo manda
el algoritmo simbólico; en PAL manda el modelo neural.


In [ ]:
respuestas = {
    "a": ("2", "control simbólico (MCTS) con subrutina neural"),
    "b": ("6", "la red controla e invoca un motor exacto"),
    "c": ("4", "regla compilada en la pérdida de entrenamiento"),
    "d": ("1", "texto → red → texto, sin componente simbólico"),
}
for k, (tipo, why) in respuestas.items():
    print(f"{k}) tipo {tipo}: {why}")


## Solución 2 — Percepción + regla

Mundos y probabilidades (independencia):

```text
(1,4) suma 5  impar  p = 0.6 × 0.55 = 0.33
(1,9) suma 10 par    p = 0.6 × 0.45 = 0.27
(7,4) suma 11 impar  p = 0.4 × 0.55 = 0.22
(7,9) suma 16 par    p = 0.4 × 0.45 = 0.18
```

Válidos: (1,9) y (7,9), masa total 0.45. Renormalizando:
P(1,9) = 0.27/0.45 = **0.6**, P(7,9) = 0.18/0.45 = **0.4**.

La lectura más probable pasa a ser (1,9). Sin restricción el argmax era (1,4),
que viola la regla: la restricción cambió la conclusión. Nota que aquí quedan
dos mundos válidos, así que la conclusión es una distribución, no una certeza.


In [ ]:
p_a = {1: 0.6, 7: 0.4}
p_b = {4: 0.55, 9: 0.45}
mundos = {(a, b): pa * pb for a, pa in p_a.items() for b, pb in p_b.items()}
validos = {m: p for m, p in mundos.items() if sum(m) % 2 == 0}
z = sum(validos.values())
renormalizados = {m: round(p / z, 4) for m, p in validos.items()}
print("mundos:", mundos)
print("válidos renormalizados:", renormalizados)
assert max(renormalizados, key=renormalizados.get) == (1, 9)


## Solución 3 — La regla del laboratorio

a) Pseudocódigo: `aceptar(claim) si claim.evidence no es None` (equivalente:
   excluir toda afirmación con `maturity == "unverified"`).

b) Queda excluida `"adopción universal"` porque su campo `evidence` es `null`:
   no hay fuente inspeccionable que la respalde.

c) Es verificable por construcción porque la decisión es una función determinista
   y legible del dato de entrada: cualquiera puede re-ejecutar la regla y obtener
   el mismo resultado, y la razón de la exclusión (evidencia ausente) está en el
   propio output. Una red entrenada daría una puntuación sin una razón simbólica
   inspeccionable, y podría cambiar al reentrenar.


In [ ]:
result = run_lab("frontier", seed=169)
aceptadas = {c["claim"] for c in result["result"]["accepted_for_curriculum"]}
todas = {c["claim"] for c in result["result"]["claims"]}
excluidas = todas - aceptadas
print("excluidas:", excluidas)
assert excluidas == {"adopción universal"}
# La regla simbólica reconstruida:
regla = [c["claim"] for c in result["result"]["claims"] if c["evidence"] is not None]
assert set(regla) == aceptadas


## Solución 4 — Cuándo el híbrido empeora

Ejemplo: un sistema de lectura de recetas médicas con la regla dura "la dosis
diaria nunca supera 4 comprimidos". Si aparece un tratamiento legítimo de 6
comprimidos, la regla —escrita con conocimiento incompleto del dominio— forzaría
al sistema a "corregir" una percepción correcta hacia una lectura errónea, con
total confianza. El supuesto que falla es que **la regla es verdadera en todos
los casos**: condicionar sobre una regla falsa amplifica el error en lugar de
corregirlo. Antes de imponer una restricción dura hay que validar su cobertura
real; si tiene excepciones, debe ser una restricción blanda (penalización) o
disparar abstención y revisión humana, no una corrección silenciosa.
